# Week 6 Lab: Meeting MODFLOW

**HWRS 564a · Fall 2026**

MODFLOW solves the groundwater flow equation on a grid. FloPy is the Python
library that writes MODFLOW input files, runs the program, and reads its output.
In this course, you will use FloPy to generate input files and inspect them when
troubleshooting a model.

In this lab, you will build a simple model, run it, compare its results with an
analytic solution, and check its water budget.

## How to use this notebook

Run each cell with **Shift+Enter**. Complete the code in cells marked
**`# YOUR TURN`**. Cells marked **`# CHECK`** test your answers against the
expected results.

## Learning objectives

By the end of this notebook you can:

1. Explain the roles of the MODFLOW packages used in this lab
2. Check that the MODFLOW executable is available
3. Build, write, and run a simple model with FloPy
4. Check `success` and capture diagnostic output from a failed run
5. Read heads from the binary `.hds` file
6. Read the water budget in the `.list` file and check its discrepancy

---

## Part 1: Check for the MODFLOW executable

FloPy runs MODFLOW as a separate compiled program. `postbuild.sh` downloads the
executable when your codespace is created. **Check that it is available** before
building the model so a missing executable is easy to diagnose.

In [ ]:
from pathlib import Path

import flopy
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path("../..")  # course repo root
MF_EXE = ROOT / "modflow" / "mf2005"

assert MF_EXE.exists(), (
    f"MODFLOW binary not found at {MF_EXE}. Your codespace may not have "
    "finished building. Run ./postbuild.sh from a terminal."
)

print(f"flopy {flopy.__version__}")
print(f"MODFLOW at {MF_EXE.relative_to(ROOT)}")
print(f"other binaries available: {sorted(p.name for p in MF_EXE.parent.iterdir())[:8]}")

Include this assertion at the start of each MODFLOW notebook. It reports a
missing executable and explains how to obtain it.

### Where the model files go

A MODFLOW run writes input and output files. **Give each model its own scratch
directory** under `_run/`, which is excluded from version control. This keeps
`.hds`, `.cbc`, and `.list` files out of the repository.

In [ ]:
WS = ROOT / "_run" / "week06_first"
WS.mkdir(parents=True, exist_ok=True)

print(f"model workspace: {WS.relative_to(ROOT)}")
print("(_run/ is excluded from version control by .gitignore)")

---

## Part 2: MODFLOW packages

MODFLOW uses separate packages to define the parts of a model. The `.nam` file
lists the packages used in a simulation.

| Model component | Package | FloPy class |
|---|---|---|
| Grid dimensions, units, and time periods | `DIS` | `ModflowDis` |
| Active cells and starting heads | `BAS` | `ModflowBas` |
| Aquifer hydraulic properties | `LPF` | `ModflowLpf` |
| Numerical solver | `PCG` | `ModflowPcg` |
| Output to save | `OC` | `ModflowOc` |
| Wells, recharge, and evapotranspiration | `WEL`, `RCH`, `EVT`, … | One class per package |

We will use `DIS`, `BAS`, `LPF`, `PCG`, and `OC` here. Later labs add stress
packages for processes such as pumping and recharge.

### A simple confined model

The model has one layer and a grid of 10 rows by 20 columns. Constant heads in
the west and east boundary columns drive flow from west to east. There are no
wells or recharge.

In [ ]:
NLAY, NROW, NCOL = 1, 10, 20
DELR = DELC = 100.0        # m, cell size

mf = flopy.modflow.Modflow("first", model_ws=str(WS), exe_name=str(MF_EXE))

# DIS: geometry and time
flopy.modflow.ModflowDis(
    mf, NLAY, NROW, NCOL,
    delr=DELR, delc=DELC,
    top=100.0, botm=0.0,       # top and bottom elevations (m)
    nper=1, steady=True,
)
print(mf.dis)

`top=100.0` and `botm=0.0` are **elevations above datum**, giving an aquifer
thickness of 100 m. Check the difference between `top` and `botm` when defining
each layer.

In [ ]:
# BAS: active cells and starting heads
ibound = np.ones((NLAY, NROW, NCOL), dtype=int)
ibound[:, :, 0] = -1        # west edge: constant head
ibound[:, :, -1] = -1       # east edge: constant head

strt = np.full((NLAY, NROW, NCOL), 95.0)
strt[:, :, -1] = 85.0       # 10 m lower on the east side

flopy.modflow.ModflowBas(mf, ibound=ibound, strt=strt)

print("ibound values used:", np.unique(ibound))
print("  1 = active,  0 = inactive,  -1 = constant head")

> **The role of `strt` depends on `ibound`.** For this steady-state model, it
> provides the solver's initial head estimates in active cells. In cells with
> `ibound == -1`, it specifies the constant boundary heads.

In [ ]:
# LPF: hydraulic properties. PCG: solver. OC: output to save.
flopy.modflow.ModflowLpf(mf, hk=10.0, laytyp=0, ipakcb=53)
flopy.modflow.ModflowPcg(mf)
flopy.modflow.ModflowOc(
    mf, stress_period_data={(0, 0): ["save head", "save budget", "print budget"]}
)

print("packages attached:", mf.get_package_list())

Two package arguments used in this lab:

- **`laytyp=0`** specifies a confined layer with fixed transmissivity.
  **`laytyp=1`** specifies a convertible layer, where transmissivity can depend
  on the computed head. We will examine this behavior in Week 8.
- **`ipakcb=53`** tells the package to write its flows to the cell-budget file.
  Include it to save that package's budget terms.

### YOUR TURN 1

Calculate the expected head gradient and specific discharge before running the
model.

The two constant-head boundaries are 95 m and 85 m, and there are 20 columns of
100 m cells. With no recharge or wells, the head profile should be a straight
line between the boundaries.

Make a **first estimate** using the full width of the domain:

- `domain_length_m`: the east–west extent of the model
- `expected_gradient`: the dimensionless hydraulic gradient
- `expected_q`: the specific discharge in m/d, from Darcy's law with `hk=10.0`

In Part 5, you will compare this estimate with MODFLOW's flow rate and account
for the positions of the boundary cell centres.

In [ ]:
hk = 10.0

# YOUR TURN
domain_length_m = ...
expected_gradient = ...
expected_q = ...

In [ ]:
# CHECK
assert abs(domain_length_m - 2000.0) < 1e-9, f"got {domain_length_m}"
assert abs(expected_gradient - 0.005) < 1e-9, f"got {expected_gradient}"
assert abs(expected_q - 0.05) < 1e-9, f"got {expected_q}"
print(f"domain      {domain_length_m:.0f} m")
print(f"gradient    {expected_gradient:.4f}")
print(f"q           {expected_q:.3f} m/d")
print("\nIn Part 5, compare this estimate with the MODFLOW budget.")

---

## Part 3: Run the model and check its status

`write_input()` writes the FloPy model's input files. `run_model()` runs the
MODFLOW executable.

In [ ]:
mf.write_input()

for f in sorted(WS.iterdir()):
    print(f"  {f.name:16s} {f.stat().st_size:8,d} bytes")

In [ ]:
success, buff = mf.run_model(silent=True, report=True)

assert success, "MODFLOW did not converge:\n" + "\n".join(buff[-20:])
print(f"converged. {len(buff)} lines of output captured.")

> **Check the run status and capture its output.**
>
> `run_model()` returns `(success, buff)`. Check `success` before reading the
> results; files left from an earlier run can otherwise be mistaken for new
> output.
>
> Use **`report=True`** to capture MODFLOW output in `buff`. With `silent=True`
> alone, `buff` is empty, so the assertion has no diagnostic output to include.
>
> **`silent=True`** suppresses console output. The `.list` file still contains
> the run log and details about convergence and the water budget.

### YOUR TURN 2: Examine a failed run

Use restrictive solver settings to produce a convergence failure and inspect
the output.

The code below rebuilds the model with `laytyp=1` and limits the solver to one
outer iteration and one inner iteration. The head and residual tolerances are
both `1e-12`.

Capture `success` and `buff` **without asserting** so you can examine the failed
run in the following cells.

In [ ]:
WS_FAIL = ROOT / "_run" / "week06_fail"
WS_FAIL.mkdir(parents=True, exist_ok=True)

bad = flopy.modflow.Modflow("nope", model_ws=str(WS_FAIL), exe_name=str(MF_EXE))
flopy.modflow.ModflowDis(bad, NLAY, NROW, NCOL, delr=DELR, delc=DELC,
                         top=100.0, botm=0.0, nper=1, steady=True)
flopy.modflow.ModflowBas(bad, ibound=ibound, strt=strt)
flopy.modflow.ModflowLpf(bad, hk=hk, laytyp=1, ipakcb=53)
flopy.modflow.ModflowPcg(bad, mxiter=1, iter1=1, hclose=1e-12, rclose=1e-12)
flopy.modflow.ModflowOc(bad, stress_period_data={(0, 0): ["save head"]})
bad.write_input()

# YOUR TURN
bad_success, bad_buff = ...

In [ ]:
# CHECK
assert bad_success is False, f"expected a convergence failure (got {bad_success})"
assert len(bad_buff) > 0, (
    "buff is empty. Use report=True to capture diagnostic output."
)
print(f"success = {bad_success}, with {len(bad_buff)} lines of output")
print("\nlast few lines:")
for line in [l.strip() for l in bad_buff if l.strip()][-3:]:
    print(f"  {line}")

Look for **FAILED TO MEET SOLVER CONVERGENCE CRITERIA** in the output.

The `.list` file contains additional diagnostics, including the water-budget
discrepancy:

In [ ]:
listing = (WS_FAIL / "nope.list").read_text().splitlines()
tail = [l.strip() for l in listing if l.strip()]

for line in tail[-6:]:
    print(line)

The failed run reports a budget discrepancy of about **-162.7%**, indicating a
large imbalance between inflow and outflow. Check both solver convergence and
budget discrepancy when evaluating a run. Part 5 examines the budget in more
detail.

---

## Part 4: Read the simulated heads

MODFLOW writes heads to a binary file. Use `flopy.utils.HeadFile` to read it.

In [ ]:
hds = flopy.utils.HeadFile(str(WS / "first.hds"))

print(f"times available: {hds.get_times()}")
head = hds.get_data()                     # last time step by default
print(f"shape: {head.shape}   (layers, rows, columns)")
print(f"range: {head.min():.2f} to {head.max():.2f} m")

### YOUR TURN 3

Check whether the simulated heads follow the expected linear profile from
Part 2.

Extract the head profile along **row 5** and compare it with a straight line
between its two ends.

- `profile`: the heads along row 5 of layer 0, as a 1D array
- `max_deviation`: the largest absolute difference between `profile` and a
  straight line joining `profile[0]` to `profile[-1]`

Use `np.linspace(a, b, n)` to construct the comparison line.

In [ ]:
# YOUR TURN
profile = ...
max_deviation = ...

In [ ]:
# CHECK
assert profile.shape == (20,), f"expected 20 values, got {profile.shape}"
assert abs(profile[0] - 95.0) < 1e-6 and abs(profile[-1] - 85.0) < 1e-6, \
    "the ends should sit exactly on the constant heads"
assert max_deviation < 1e-4, f"expected a straight line, deviation was {max_deviation}"
print(f"profile ends: {profile[0]:.2f} -> {profile[-1]:.2f} m")
print(f"max deviation from a straight line: {max_deviation:.2e} m")
print("\nThe simulated heads match the linear profile within the check tolerance.")

**Test a simple model against an analytic solution** before adding more
complex processes.

A one-dimensional confined aquifer between two fixed heads has a linear head
profile. Agreement with that profile checks the grid, boundary heads, and
solver for this case. As you add processes, check their effects and investigate
unexpected changes in the results.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

mv = flopy.plot.PlotMapView(model=mf, ax=axes[0])
band = mv.plot_array(head[0], cmap="Blues_r")
cs = mv.contour_array(head[0], levels=np.arange(86, 96, 1.0), colors="k", linewidths=0.7)
axes[0].clabel(cs, fmt="%.0f", fontsize=8)
axes[0].set_xlabel("easting (m)")
axes[0].set_ylabel("northing (m)")
axes[0].set_title("Head, plan view")
fig.colorbar(band, ax=axes[0], label="head (m)", shrink=0.85)

x = (np.arange(NCOL) + 0.5) * DELR
axes[1].plot(x, profile, "o-", color="#AB0520", ms=4, label="MODFLOW")
axes[1].plot(x, np.linspace(profile[0], profile[-1], NCOL), "--",
             color="#0C234B", lw=1.5, label="analytic (linear)")
axes[1].set_xlabel("distance east (m)")
axes[1].set_ylabel("head (m)")
axes[1].set_title("Row 5 profile vs. the analytic solution")
axes[1].legend(frameon=False)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

## Part 5: The water budget

MODFLOW writes a water budget to the `.list` file. **Inspect it after each run.**
Check that the inflows and outflows balance, even when the solver reports
convergence.

In [ ]:
listing = (WS / "first.list").read_text().splitlines()

start = next(i for i, l in enumerate(listing) if "VOLUMETRIC BUDGET" in l)
for line in listing[start:start + 26]:
    if line.strip():
        print(line.rstrip())

The budget lists flows under `IN` and `OUT`. **PERCENT DISCREPANCY** reports the
imbalance between total inflow and total outflow.

In [ ]:
discrepancy = next(
    float(l.split()[-1]) for l in reversed(listing) if "PERCENT DISCREPANCY" in l
)
print(f"percent discrepancy: {discrepancy:.4f} %")

assert abs(discrepancy) < 0.1, "a converged steady-state model should close to well under 0.1%"
print("The budget closes. Water in equals water out.")

### YOUR TURN 4

Write `budget_discrepancy(list_path)` to return the percent discrepancy from a
MODFLOW `.list` file as a float.

Read the file, find the **last** line containing `"PERCENT DISCREPANCY"`, and
extract the number at the end. You will reuse this function in later labs.

MODFLOW writes the discrepancy in two formats:

```
 PERCENT DISCREPANCY =        -162.74     PERCENT DISCREPANCY =        -162.74
 BUDGET PERCENT DISCREPANCY IS -162.7401
```

**Take the last whitespace-separated token** and convert it to a float. This
handles both formats. Splitting on `"="` only handles the first.

In [ ]:
# YOUR TURN
def budget_discrepancy(list_path):
    """Percent mass-balance discrepancy from a MODFLOW .list file."""
    ...

In [ ]:
# CHECK
good = budget_discrepancy(WS / "first.list")
assert isinstance(good, float), f"should return a float, got {type(good)}"
assert abs(good) < 0.1, f"the converged model should close: got {good}"

bad_value = budget_discrepancy(WS_FAIL / "nope.list")
assert abs(bad_value) > 100, f"the failed model was off by 160%, got {bad_value}"

print(f"converged model : {good:+.4f} %")
print(f"failed model    : {bad_value:+.4f} %")
print("Correct.")

### YOUR TURN 5: Calculate the volumetric flow

In this model, the `CONSTANT HEAD` budget term reports the flow through the
system. Compare it with the Darcy estimate from Part 2.

- `cross_section_area`: the full north–south width times the aquifer thickness
- `darcy_flow`: `expected_q` times that area

In [ ]:
thickness_m = 100.0

# YOUR TURN
cross_section_area = ...
darcy_flow = ...

In [ ]:
# CHECK
assert abs(cross_section_area - 100_000.0) < 1e-6, f"got {cross_section_area}"
assert abs(darcy_flow - 5000.0) < 1e-6, f"got {darcy_flow}"

chd = next(
    float(l.split()[-1]) for l in listing if l.strip().startswith("CONSTANT HEAD =")
)
print(f"Darcy's law (Part 2) says {darcy_flow:9.1f} m3/d")
print(f"MODFLOW's budget says     {chd:9.1f} m3/d")
print(f"difference                {100 * (chd - darcy_flow) / darcy_flow:+9.1f} %")
print("\nThe flow estimates differ by about 5%. Check the boundary positions below.")

### The distance between boundary heads

The first estimate used the full 2,000 m domain width. MODFLOW specifies
**constant heads at cell centres**: 95 m at the centre of column 0 and 85 m at
the centre of column 19. The distance between them is

$$19 	imes 100\ 	ext{m} = 1900\ 	ext{m}$$

Each boundary cell centre sits half a cell width inside the model edge. Use the
1,900 m distance between centres to calculate the gradient.

### YOUR TURN 6

Recalculate specific discharge and volumetric flow using that distance.

- `centre_to_centre_m`: the distance between the constant-head cell centres
- `corrected_q`: specific discharge over that distance
- `corrected_flow`: volumetric flow through the cross section

In [ ]:
# YOUR TURN
centre_to_centre_m = ...
corrected_q = ...
corrected_flow = ...

In [ ]:
# CHECK
assert abs(centre_to_centre_m - 1900.0) < 1e-9, f"got {centre_to_centre_m}"
assert abs(corrected_flow - chd) < 0.1, (
    f"expected to match MODFLOW's {chd:.2f}, got {corrected_flow:.2f}"
)
print(f"centre-to-centre distance {centre_to_centre_m:9.0f} m")
print(f"corrected q               {corrected_q:9.5f} m/d")
print(f"corrected flow            {corrected_flow:9.2f} m3/d")
print(f"MODFLOW                   {chd:9.2f} m3/d")
print("\nThe corrected flow matches the MODFLOW budget within the check tolerance.")

Comparing the budget with Darcy's law revealed a difference in the distance
used to calculate the gradient. Accounting for the boundary cell centres brings
the two flow estimates into agreement.

**Constant heads apply at the boundary cell centres.** On a 250 m grid, each
centre is 125 m inside the model edge. Use those positions when comparing
modeled and observed gradients.

You have checked this model against a linear head profile, inspected its water
budget, and compared its flow rate with Darcy's law. Use these results as a
baseline when adding new processes.

Next week, you will add a well and recharge and examine how they change the
head field.

---

## Before you leave

1. *Kernel → Restart Kernel and Run All Cells*
2. Fix anything that breaks
3. Save

## What's due

- **HW 3: From labelled data to a verified first model**, Wednesday 10/7 at 11:59pm

## Next week

Build a basin model with a well and recharge, then examine the resulting head
field.

## Stuck?

- `FileNotFoundError` mentioning `mf2005`: the binary isn't there. Run
  `./postbuild.sh` from a terminal.
- `success` is `False` and `buff` is empty: you left out `report=True`.
- A model that converges but whose heads are all identical to `strt` usually
  means every cell is `ibound == -1`, so nothing was free to change.
- `AssertionError` on layer thickness, or a `.list` file complaining about
  `BOTM`, means `top` and `botm` are the wrong way round.
- Office hours: Tuesdays 1:00–2:00pm, Harshbarger 322B.